# EDAI: FLAN-T5 구조 생성 학습

스펙 문장을 넣으면 OP-Amp 구조 표를 만드는 모델을 학습한다.

순서
1. 런타임 > 런타임 유형 변경 > **T4 GPU** 를 고른다
2. 아래 셀을 위에서부터 차례로 실행한다. 2번 셀에서 `train.jsonl`, `val.jsonl`, `tlv271.json` 세 파일을 올린다 (저장소의 `data/train/` 폴더, `python make_train_data.py` 로 만든 것)
3. 마지막 셀이 `tlv271_candidates.json` 을 내려받는다. 이 파일을 저장소의 `data/train/` 에 넣으면 크기 정하기와 시뮬레이션으로 후보를 고른다

학습은 T4 기준 30~40분 걸린다.

In [ ]:
# 1. GPU 확인과 설치
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U transformers datasets accelerate sentencepiece

In [ ]:
# 2. 학습 파일 올리기 (train.jsonl, val.jsonl, tlv271.json)
from google.colab import files
uploaded = files.upload()
for name in ("train.jsonl", "val.jsonl", "tlv271.json"):
    assert name in uploaded, f"{name} 을 올려야 한다"
print("올린 파일:", list(uploaded))

In [ ]:
# 3. 설정
MODEL_NAME = "google/flan-t5-base"   # 2.5억 개 가중치. T4 에서 학습 가능한 크기
MAX_INPUT_TOKENS = 256
MAX_TARGET_TOKENS = 768
EPOCHS = 20
LEARNING_RATE = 3e-4
BATCH_SIZE = 8
SEED = 7
OUTPUT_DIR = "flan_t5_structure"
ROW_SEP = " ; "


In [ ]:
# 4. 데이터 읽기, 토큰 길이와 글자 손실 확인
import json, collections
from transformers import AutoTokenizer

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_rows, val_rows = read_jsonl("train.jsonl"), read_jsonl("val.jsonl")
tlv = json.load(open("tlv271.json", encoding="utf-8"))
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def normalize(text):
    return " ".join(text.split())

lengths = [len(tokenizer(r["target"]).input_ids) for r in train_rows + val_rows]
lost = [r["target"] for r in train_rows[:200]
        if normalize(tokenizer.decode(tokenizer(r["target"]).input_ids, skip_special_tokens=True)) != normalize(r["target"])]
print(f"학습 {len(train_rows)}개, 검증 {len(val_rows)}개")
print(f"정답 토큰 수: 평균 {sum(lengths)/len(lengths):.0f}, 최대 {max(lengths)} (자르는 기준 {MAX_TARGET_TOKENS})")
print(f"토큰으로 바꿨다가 되돌렸을 때 글자가 바뀌는 정답: {len(lost)} / 200")
assert max(lengths) <= MAX_TARGET_TOKENS, "MAX_TARGET_TOKENS 를 늘린다"
print("TLV271 입력:", tlv["input"])

In [ ]:
# 5. 구조 표 검사 (모델이 쓴 글이 회로로 말이 되는지)
import re
PINS = {"NM": ("D", "G", "S", "B"), "PM": ("D", "G", "S", "B"), "R": ("P", "N"), "C": ("P", "N")}
PORTS = ("VDD", "VSS", "IN1", "IN2", "OUT")

def parse_table(text):
    rows = []
    for part in text.split(";"):
        tokens = part.split()
        if not tokens:
            continue
        match = re.fullmatch(r"(NM|PM|R|C)(\d+)", tokens[0])
        if not match:
            raise ValueError(f"소자 이름이 이상함: {tokens[0]}")
        pins = dict(t.split("=", 1) for t in tokens[1:] if "=" in t)
        if tuple(pins) != PINS[match.group(1)]:
            raise ValueError(f"{tokens[0]} 의 단자가 이상함: {list(pins)}")
        rows.append((tokens[0], pins))
    if not rows:
        raise ValueError("빈 표")
    return rows

def problems(text):
    """고쳐야 할 점 목록. 빈 목록이면 통과."""
    try:
        rows = parse_table(text)
    except ValueError as error:
        return [f"형식: {error}"]
    found = []
    names = [name for name, _ in rows]
    if len(set(names)) != len(names):
        found.append("같은 이름의 소자가 있음")
    count = collections.Counter(net for _, pins in rows for net in pins.values())
    for port in PORTS:
        if port not in count:
            found.append(f"{port} 단자가 없음")
    gates = {pins["G"] for _, pins in rows if "G" in pins}
    for port in ("IN1", "IN2"):
        if port not in gates:
            found.append(f"{port} 가 트랜지스터 게이트에 붙지 않음")
    floating = [net for net, n in count.items() if n < 2 and net not in PORTS and not net.startswith(("VB", "IB"))]
    if floating:
        found.append(f"한 곳에만 붙은 노드: {floating}")
    return found

bad = [r for r in train_rows if problems(r["target"])]
print(f"학습 정답 중 검사에 걸리는 것: {len(bad)} (0 이어야 검사가 믿을 만함)")
if bad:
    print(problems(bad[0]["target"]))

In [ ]:
# 6. 학습
import torch
from datasets import Dataset
from transformers import (AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments, set_seed)
set_seed(SEED)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def encode(batch):
    enc = tokenizer(batch["input"], max_length=MAX_INPUT_TOKENS, truncation=True)
    enc["labels"] = tokenizer(text_target=batch["target"], max_length=MAX_TARGET_TOKENS, truncation=True).input_ids
    return enc

train_ds = Dataset.from_list(train_rows).map(encode, batched=True, remove_columns=["input", "target", "structure"])
val_ds = Dataset.from_list(val_rows).map(encode, batched=True, remove_columns=["input", "target", "structure"])
args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2, load_best_model_at_end=True,
    metric_for_best_model="eval_loss", logging_steps=50, report_to="none", seed=SEED,
    fp16=False,   # T5 는 fp16 에서 값이 터지기 쉬워 fp32 로 학습한다
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                         data_collator=DataCollatorForSeq2Seq(tokenizer, model=model))
trainer.train()
trainer.save_model(OUTPUT_DIR + "/best")
tokenizer.save_pretrained(OUTPUT_DIR + "/best")
history = [h for h in trainer.state.log_history if "eval_loss" in h]
print("에폭별 검증 손실:", [round(h["eval_loss"], 4) for h in history])

In [ ]:
# 7. 검증 구조로 평가 (학습에 없던 구조를 얼마나 만드나)
model.eval()
device = model.device
train_targets = {normalize(r["target"]) for r in train_rows}

def generate(texts, **kwargs):
    enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_INPUT_TOKENS).to(device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_TARGET_TOKENS, **kwargs)
    return [normalize(t) for t in tokenizer.batch_decode(out, skip_special_tokens=True)]

full_inputs = {}
for r in val_rows:          # 구조마다 항목이 가장 많은 문장 하나로 평가
    if len(r["input"]) > len(full_inputs.get(r["target"], {"input": ""})["input"]):
        full_inputs[r["target"]] = r
picked = list(full_inputs.values())
outputs = []
for i in range(0, len(picked), 8):
    outputs += generate([r["input"] for r in picked[i:i + 8]], num_beams=4)
stats = collections.Counter()
for r, out in zip(picked, outputs):
    stats["형식 통과"] += not any(p.startswith("형식") for p in problems(out))
    stats["검사 모두 통과"] += not problems(out)
    stats["정답과 같음"] += out == normalize(r["target"])
    stats["학습 데이터에 있는 구조"] += out in train_targets
n = len(picked)
for key, value in stats.items():
    print(f"{key}: {value} / {n} ({100 * value / n:.0f}%)")
print("\n예시 (입력 -> 생성):")
print(picked[0]["input"]); print(outputs[0]); print(problems(outputs[0]) or "검사 통과")

In [ ]:
# 8. TLV271 후보 만들기 -> tlv271_candidates.json 내려받기
CANDIDATES_BEAM = 10
CANDIDATES_SAMPLED = 30
beams = generate([tlv["input"]], num_beams=CANDIDATES_BEAM, num_return_sequences=CANDIDATES_BEAM)
sampled = []
for _ in range(CANDIDATES_SAMPLED // 10):
    sampled += generate([tlv["input"]], do_sample=True, top_p=0.95, temperature=0.8, num_return_sequences=10)
candidates = []
for how, text in [("beam", t) for t in beams] + [("sample", t) for t in sampled]:
    if any(c["table"] == text for c in candidates):
        continue
    candidates.append({"how": how, "table": text, "problems": problems(text), "in_train": text in train_targets})
passed = [c for c in candidates if not c["problems"]]
print(f"서로 다른 후보 {len(candidates)}개, 검사 통과 {len(passed)}개, 그중 학습 데이터에 없던 구조 {sum(not c['in_train'] for c in passed)}개")
result = {"model": MODEL_NAME, "epochs": EPOCHS, "input": tlv["input"], "eval": dict(stats), "eval_count": n,
          "candidates": candidates}
json.dump(result, open("tlv271_candidates.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
files.download("tlv271_candidates.json")

In [ ]:
# 9. (선택) 모델을 구글 드라이브에 저장. 다시 학습하지 않고 후보만 새로 만들 때 쓴다
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/EDAI
!cp -r {OUTPUT_DIR}/best /content/drive/MyDrive/EDAI/flan_t5_structure